In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

from transformer import ThermalTransformerRegressor
from loader import load_data

In [4]:
holdout = None
#holdout = [-12.502,-29.500,-41.010]

train_loader,train_val_loader,test_loader = load_data('data/full_dataset/data_rearranged.csv',holdout=holdout,batch_size=16)

In [10]:
base_lr = 1e-4
thermal_lr = 8e-2
hidden_dim = 200

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ThermalTransformerRegressor(input_dim=5,d_model=hidden_dim,nhead=5).to(device).float()
criterion = nn.MSELoss()

thermal_params = [p for n, p in model.named_parameters() if 'log_beta' in n or 'mu' in n]
base_params = [p for n, p in model.named_parameters() if 'log_beta' not in n and 'mu' not in n]

optimizer = torch.optim.Adam([
    {'params': base_params, 'lr': base_lr}, 
    {'params': thermal_params, 'lr': thermal_lr}
])

num_epochs = 100

print(f"Training on {device}...")

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device).float(), batch_y.to(device).float()
        
        preds = model(batch_x)
        mse_loss = criterion(preds, batch_y)
        rmse_loss = torch.sqrt(mse_loss + 1e-15)
        
        optimizer.zero_grad()
        rmse_loss.backward()
        optimizer.step()
        
        total_train_loss += rmse_loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    model.eval()
    total_test_loss = 0
    
    with torch.no_grad():
        for batch_x, batch_y in train_val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            preds = model(batch_x)
            mse_loss = criterion(preds, batch_y)
            rmse_loss = torch.sqrt(mse_loss + 1e-15)
            total_test_loss += rmse_loss.item()
            
    avg_test_loss = total_test_loss / len(test_loader)
    
    # --- LOGGING ---
    if (epoch + 1) % 5 == 0 or epoch == 0:
        sample_beta = np.exp([x.log_beta.item() for x in model.heads])
        sample_mu = np.exp([x.mu.item() for x in model.heads])
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train RMSE: {avg_train_loss:.4f} | Test RMSE: {avg_test_loss:.4f} | Beta: {sample_beta} | Mu: {sample_mu}")
        print("-" * 30)

model.eval()
total_test_loss = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        preds = model(batch_x)
        mse_loss = criterion(preds, batch_y)
        rmse_loss = torch.sqrt(mse_loss + 1e-15)
        total_test_loss += rmse_loss.item()
        
avg_test_loss = total_test_loss / len(test_loader)
print(f"Final RMSE: {avg_test_loss}")

Training on cuda...
Epoch [1/100]
  Train RMSE: 3.9518 | Test RMSE: 2.2442 | Beta: [  0.40227887   0.65655599   0.68494848 119.8019586    6.08459487] | Mu: [1.28758204 1.59299852 0.09922475 0.36953677 0.19893053]
------------------------------
Epoch [5/100]
  Train RMSE: 1.5087 | Test RMSE: 2.2230 | Beta: [  0.39577142   0.38621201   0.90789495 112.77602653   6.5844224 ] | Mu: [0.61173094 1.52939977 0.07156185 0.41731542 0.9537069 ]
------------------------------
Epoch [10/100]
  Train RMSE: 1.2713 | Test RMSE: 1.2795 | Beta: [  0.30560995   0.44113593   0.73722808 112.79850705   7.35848113] | Mu: [0.46812205 0.58939859 3.87151738 0.4171175  0.95363777]
------------------------------
Epoch [15/100]
  Train RMSE: 1.0341 | Test RMSE: 1.1135 | Beta: [  0.41463515   0.37145144   1.1365794  112.8785698    5.33585156] | Mu: [ 4.85729418  1.49514719 66.74564893  0.41642311  0.95387239]
------------------------------
Epoch [20/100]
  Train RMSE: 0.9433 | Test RMSE: 0.9737 | Beta: [  0.29062456

In [11]:
# Specify a path
from datetime import datetime

now = datetime.now()
formatted_time = now.strftime("%d%B%Y-%H%M")
out_path = f"thermal_transformer_hidden-{hidden_dim}_base-{base_lr}_thermal-{thermal_lr}_{formatted_time}.pth"

torch.save(model.state_dict(), out_path)
print(f"Model parameters saved to {out_path}")

Model parameters saved to thermal_transformer_hidden-200_base-0.0001_thermal-0.08_26April2026-2332.pth


In [ ]:
model = ThermalTransformerRegressor(input_dim=5, d_model=64, nhead=8)
model.load_state_dict(torch.load("thermal_transformer_26April2026-2150.pth"))
model.eval()

total_test_loss = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        preds = model(batch_x)
        mse_loss = criterion(preds, batch_y)
        rmse_loss = torch.sqrt(mse_loss + 1e-15)
        total_test_loss += rmse_loss.item()
        
avg_test_loss = total_test_loss / len(test_loader)
print(f"Final MSE: {avg_test_loss}")